In [1]:
import argparse
import math
import pandas as pd

def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = successes / n
    denom = 1 + (z**2)/n
    center = (p + (z**2)/(2*n)) / denom
    margin = z * math.sqrt((p*(1-p) + (z**2)/(4*n)) / n) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))

def anaylse(csv_path):
    df = pd.read_csv(csv_path, sep=None, engine="python")

    # Coerce types
    # win column sometimes comes as TRUE/FALSE strings; normalize to bool
    if df["win"].dtype == object:
        df["win"] = df["win"].astype(str).str.lower().isin(["true", "1", "yes"])
    else:
        df["win"] = df["win"].astype(bool)

    # If your runner logs only the last step per episode (typical), you can skip grouping.
    # Otherwise, keep only rows with episode aggregates present:
    if "ep_intent_total" in df.columns:
        df_term = df[df["ep_intent_total"].fillna(0) > 0].copy()
    else:
        # Fallback: take the last row per episode
        if "episode" in df.columns:
            df_term = df.sort_values(["episode", "total_steps"]).groupby("episode").tail(1).copy()
        else:
            df_term = df.copy()


    # --- Metrics ---
    n_ep = len(df_term)
    wins = int(df_term["win"].sum())
    win_rate = wins / n_ep if n_ep > 0 else 0.0
    wl = wilson_interval(wins, n_ep)

    imit_acc = df_term["ep_imitation_accuracy"].mean() if "ep_imitation_accuracy" in df_term.columns else float("nan")
    turns_win = df_term.loc[df_term["win"], "ep_turns"]
    mean_turns_win = turns_win.mean() if len(turns_win) > 0 else float("nan")

    print("=== Evaluation Summary ===")
    print(f"CSV path:                 {csv_path}")
    print(f"Episodes:                 {n_ep}")
    print(f"Wins:                     {wins}")
    print(f"Win rate:                 {win_rate*100:.2f}%  (95% CI: {wl[0]*100:.2f}–{wl[1]*100:.2f}%)")
    print(f"Wins per 100 episodes:    {win_rate*100:.2f}")
    print(f"Mean turns per win:       {mean_turns_win:.2f}")
    print(f"Imitation accuracy (ep):  {imit_acc*100:.2f}%")

# anaylse("approach-a-200000.csv")
# anaylse("approach-a-400000.csv")
# anaylse("approach-b-200000.csv")
# anaylse("approach-b-200000-take-2.csv")
anaylse("approach-b-400000.csv")
anaylse("approach-b-fixed-boosts.csv")
# anaylse("ideal-1000.csv")
# anaylse("modified-simple-huerstic-player.csv")
# anaylse("non-modified-simple-huerstic-player.csv")


=== Evaluation Summary ===
CSV path:                 approach-b-400000.csv
Episodes:                 100
Wins:                     80
Win rate:                 80.00%  (95% CI: 71.12–86.66%)
Wins per 100 episodes:    80.00
Mean turns per win:       19.74
Imitation accuracy (ep):  79.23%
=== Evaluation Summary ===
CSV path:                 approach-b-fixed-boosts.csv
Episodes:                 1000
Wins:                     766
Win rate:                 76.60%  (95% CI: 73.88–79.12%)
Wins per 100 episodes:    76.60
Mean turns per win:       19.27
Imitation accuracy (ep):  80.97%
